In [4]:
# Reload the dataset fresh
df = pd.read_csv('data/diabetic_data.csv')

# Step 1 - Replace ? with NaN
df.replace('?', np.nan, inplace=True)

# Step 2 - Keep only first encounter per patient BEFORE dropping columns
df = df.sort_values('encounter_id')
df = df.drop_duplicates(subset='patient_nbr', keep='first')

print(f"Shape after keeping first encounter: {df.shape}")
print(f"Unique patients: {df['patient_nbr'].nunique()}")

Shape after keeping first encounter: (71518, 50)
Unique patients: 71518


In [5]:
cols_to_drop = ['encounter_id', 'patient_nbr', 'weight', 
                'payer_code', 'max_glu_serum', 'A1Cresult']

df.drop(columns=cols_to_drop, inplace=True)

print(f"Columns dropped successfully!")
print(f"New shape: {df.shape}")

Columns dropped successfully!
New shape: (71518, 44)


In [6]:
# 1. Race - fill with mode (most frequent value)
df['race'].fillna(df['race'].mode()[0], inplace=True)

# 2. Medical specialty - fill with 'Unknown'
df['medical_specialty'].fillna('Unknown', inplace=True)

# 3. Diagnosis columns - fill with 'Unknown'
df['diag_1'].fillna('Unknown', inplace=True)
df['diag_2'].fillna('Unknown', inplace=True)
df['diag_3'].fillna('Unknown', inplace=True)

# Verify no missing values remain
missing = df.isnull().sum()
print("Remaining missing values:")
print(missing[missing > 0])
print("\nMissing values handled successfully!" if missing.sum() == 0 else "Some missing values remain!")

C:\Users\SOHAIL ALI J\AppData\Local\Temp\ipykernel_38332\3408632634.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['race'].fillna(df['race'].mode()[0], inplace=True)
C:\Users\SOHAIL ALI J\AppData\Local\Temp\ipykernel_38332\3408632634.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves 

Remaining missing values:
Series([], dtype: int64)

Missing values handled successfully!


In [7]:
# Modern pandas syntax - avoids FutureWarning
df = df.fillna({
    'race': df['race'].mode()[0],
    'medical_specialty': 'Unknown',
    'diag_1': 'Unknown',
    'diag_2': 'Unknown',
    'diag_3': 'Unknown'
})

# Verify
missing = df.isnull().sum()
print("Remaining missing values:")
print(missing[missing > 0])
print("\nAll missing values handled successfully!" if missing.sum() == 0 else "Some missing values remain!")

Remaining missing values:
Series([], dtype: int64)

All missing values handled successfully!


In [8]:
# Convert target column to binary
# 1 = Readmitted within 30 days (<30) - Critical case
# 0 = Not readmitted or readmitted after 30 days (NO or >30)

df['readmitted'] = (df['readmitted'] == '<30').astype(int)

# Verify
print("Target column distribution:")
print(df['readmitted'].value_counts())
print(f"\nPercentage of critical readmissions: {df['readmitted'].mean() * 100:.2f}%")

Target column distribution:
readmitted
0    65225
1     6293
Name: count, dtype: int64

Percentage of critical readmissions: 8.80%


In [9]:
# Age is in ranges like [0-10), [10-20) etc.
# Let's convert it to numeric midpoint values

age_mapping = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25,
    '[30-40)': 35, '[40-50)': 45, '[50-60)': 55,
    '[60-70)': 65, '[70-80)': 75, '[80-90)': 85,
    '[90-100)': 95
}

df['age'] = df['age'].map(age_mapping)

print("Age column after conversion:")
print(df['age'].value_counts().sort_index())

Age column after conversion:
age
5       154
15      535
25     1127
35     2699
45     6878
55    12466
65    15960
75    18210
85    11589
95     1900
Name: count, dtype: int64


In [10]:
# Check unique gender values
print("Gender values:", df['gender'].unique())

# Remove invalid/unknown gender entries
df = df[df['gender'] != 'Unknown/Invalid']

print(f"\nShape after removing invalid gender: {df.shape}")

Gender values: ['Female' 'Male' 'Unknown/Invalid']

Shape after removing invalid gender: (71515, 44)


In [11]:
# Load the mapping file
df_mapping = pd.read_csv('data/IDS_mapping.csv')

print("Mapping file columns:")
print(df_mapping.columns.tolist())
print("\nFirst few rows:")
print(df_mapping.head(10))

Mapping file columns:
['admission_type_id', 'description']

First few rows:
          admission_type_id    description
0                         1      Emergency
1                         2         Urgent
2                         3       Elective
3                         4        Newborn
4                         5  Not Available
5                         6            NaN
6                         7  Trauma Center
7                         8     Not Mapped
8                       NaN            NaN
9  discharge_disposition_id    description


In [12]:
# The mapping file has all 3 mappings stacked together
# We need to split them into separate dictionaries

# Read raw file
df_mapping = pd.read_csv('data/IDS_mapping.csv')

# Split into 3 separate mappings
# Find where each section starts
admission_type_map = {}
discharge_disposition_map = {}
admission_source_map = {}

current_map = None

for _, row in df_mapping.iterrows():
    key = str(row.iloc[0]).strip()
    val = str(row.iloc[1]).strip()
    
    if key == 'admission_type_id':
        current_map = admission_type_map
    elif key == 'discharge_disposition_id':
        current_map = discharge_disposition_map
    elif key == 'admission_source_id':
        current_map = admission_source_map
    elif key != 'nan' and val != 'nan' and current_map is not None:
        try:
            current_map[int(float(key))] = val
        except:
            pass

print("Admission Type Mapping:")
print(admission_type_map)
print("\nDischarge Disposition Mapping:")
print(discharge_disposition_map)
print("\nAdmission Source Mapping:")
print(admission_source_map)

Admission Type Mapping:
{}

Discharge Disposition Mapping:
{1: 'Discharged to home', 2: 'Discharged/transferred to another short term hospital', 3: 'Discharged/transferred to SNF', 4: 'Discharged/transferred to ICF', 5: 'Discharged/transferred to another type of inpatient care institution', 6: 'Discharged/transferred to home with home health service', 7: 'Left AMA', 8: 'Discharged/transferred to home under care of Home IV provider', 9: 'Admitted as an inpatient to this hospital', 10: 'Neonate discharged to another hospital for neonatal aftercare', 11: 'Expired', 12: 'Still patient or expected to return for outpatient services', 13: 'Hospice / home', 14: 'Hospice / medical facility', 15: 'Discharged/transferred within this institution to Medicare approved swing bed', 16: 'Discharged/transferred/referred another institution for outpatient services', 17: 'Discharged/transferred/referred to this institution for outpatient services', 19: 'Expired at home. Medicaid only, hospice.', 20: 'Expi

In [13]:
# Manually define admission type mapping from the IDS_mapping file
admission_type_map = {
    1: 'Emergency',
    2: 'Urgent',
    3: 'Elective',
    4: 'Newborn',
    5: 'Not Available',
    6: 'Not Available',
    7: 'Trauma Center',
    8: 'Not Mapped'
}

# Now apply all 3 mappings to the dataframe
df['admission_type_id'] = df['admission_type_id'].map(admission_type_map)
df['discharge_disposition_id'] = df['discharge_disposition_id'].map(discharge_disposition_map)
df['admission_source_id'] = df['admission_source_id'].map(admission_source_map)

# Rename columns for clarity
df.rename(columns={
    'admission_type_id': 'admission_type',
    'discharge_disposition_id': 'discharge_disposition',
    'admission_source_id': 'admission_source'
}, inplace=True)

# Verify
print("Admission Type values:")
print(df['admission_type'].value_counts())

Admission Type values:
admission_type
Emergency        36488
Elective         13916
Urgent           13028
Not Available     7762
Not Mapped         291
Trauma Center       21
Newborn              9
Name: count, dtype: int64


In [14]:
# Save cleaned dataset for use in next modules
df.to_csv('data/diabetic_data_cleaned.csv', index=False)

print("Cleaned dataset saved successfully!")
print(f"Final shape: {df.shape}")
print(f"\nColumn list:")
print(df.columns.tolist())

Cleaned dataset saved successfully!
Final shape: (71515, 44)

Column list:
['race', 'gender', 'age', 'admission_type', 'discharge_disposition', 'admission_source', 'time_in_hospital', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']
